# Travaille sur les filtres dans la feuille de graphiques

In [1]:
from openpyxl import load_workbook
from openpyxl import Workbook
from openpyxl.worksheet.worksheet import Worksheet

In [ ]:
wb=load_workbook("../output/test/test.xlsx")

## 1er code proposé par Claude

In [5]:
def add_dropdown_and_chart(wb: Workbook, save_as: str):
    ws_scatter = wb["SCATTER"]

    # 1. Activer le filtre automatique sur toutes les colonnes
    ws_scatter.auto_filter.ref = f"A1:D{ws_scatter.max_row}"

    # 2. Ajouter une liste déroulante sur la colonne Domains (col D)
    # Extraire les domaines uniques
    domains = set()
    for row in ws_scatter.iter_rows(min_row=2, max_row=ws_scatter.max_row, min_col=4, max_col=4):
        for cell in row:
            if cell.value:
                for d in cell.value.split(", "):
                    domains.add(d.strip())

    # Écrire les domaines dans une zone hors tableau (ex: colonne F)
    ws_scatter["F1"] = "Tous"
    for i, domain in enumerate(sorted(domains), start=2):
        ws_scatter.cell(row=i, column=6).value = domain

    # 3. Créer la liste déroulante sur D1 (l'en-tête Domains)
    from openpyxl.worksheet.datavalidation import DataValidation
    last_row = len(domains) + 1
    dv = DataValidation(
        type="list",
        formula1=f"SCATTER!$F$1:$F${last_row}",
        allow_blank=True,
        showDropDown=True
    )
    dv.sqref = "D1"
    ws_scatter.add_data_validation(dv)

    wb.save(save_as)

In [ ]:
add_dropdown_and_chart(wb, "../output/test/feat-2.xlsx")

Conclusion : ajoute une liste des valeurs dans la feuille tdb mais ne permet pas de filtrer dynamiquement

## 2ème code proposé par Claude

In [7]:
from openpyxl.worksheet.datavalidation import DataValidation

def add_dropdown_filter(wb: Workbook, save_as: str):
    ws_tdb = wb["TDB"]
    ws_scatter = wb["SCATTER"]

    # 1. Extraire les valeurs uniques de Domains depuis SCATTER
    domains = set()
    for row in ws_scatter.iter_rows(min_row=2, max_row=ws_scatter.max_row, min_col=4, max_col=4):
        for cell in row:
            if cell.value:
                # Séparer les domaines multiples (ex: "Strategy Games, Thematic Games")
                for d in cell.value.split(", "):
                    domains.add(d.strip())

    # 2. Écrire les valeurs uniques dans une zone de TDB (ex: colonne B, cachée)
    ws_tdb["A1"] = "Filtre Domaine"
    ws_tdb["B1"] = "Tous"  # option par défaut
    for i, domain in enumerate(sorted(domains), start=2):
        ws_tdb.cell(row=i, column=2).value = domain

    # 3. Créer la liste déroulante en A2 dans TDB
    # La plage source est la colonne B de TDB
    last_row = len(domains) + 1
    dv = DataValidation(
        type="list",
        formula1=f"TDB!$B$1:$B${last_row}",
        allow_blank=True,
        showDropDown=False  # False = afficher la flèche déroulante
    )
    dv.sqref = "A2"  # cellule où apparaît la liste déroulante
    ws_tdb.add_data_validation(dv)

    ws_tdb["A2"] = "Tous"  # valeur par défaut sélectionnée

    wb.save(save_as)

In [ ]:
add_dropdown_filter(wb, "../output/test/feat-2.xlsx")

Conclusion : ça ressemble au premier résultat

## 3ème code proposé par Perplexity

In [16]:
from openpyxl import load_workbook
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.workbook.defined_name import DefinedName
from openpyxl.utils import get_column_letter

file_path = "../test.xlsx"

wb = load_workbook(file_path)
ws_scatter = wb["SCATTER"]
ws_tdb = wb["TDB"]

# -----------------------------
# 1) Récupérer les valeurs uniques de Domains (colonne D)
# -----------------------------
domains = []
seen = set()

for row in range(2, ws_scatter.max_row + 1):
    value = ws_scatter[f"D{row}"].value
    if value is not None and str(value).strip() != "":
        value = str(value)
        if value not in seen:
            seen.add(value)
            domains.append(value)

domains.sort()

# -----------------------------
# 2) Écrire la liste technique dans TDB, colonne Z
# -----------------------------
start_row = 2
helper_col = 26  # Z
helper_col_letter = get_column_letter(helper_col)

ws_tdb[f"{helper_col_letter}1"] = "LISTE_DOMAINS"
for i, domain in enumerate(domains, start=start_row):
    ws_tdb[f"{helper_col_letter}{i}"] = domain

end_row = start_row + len(domains) - 1

# -----------------------------
# 3) Créer / remplacer le nom défini LISTE_DOMAINS
# -----------------------------
if "LISTE_DOMAINS" in wb.defined_names:
    del wb.defined_names["LISTE_DOMAINS"]

ref_domains = f"'TDB'!${helper_col_letter}$2:${helper_col_letter}${end_row}"
wb.defined_names["LISTE_DOMAINS"] = DefinedName("LISTE_DOMAINS", attr_text=ref_domains)

# -----------------------------
# 4) Créer la dropdown en TDB!A2
# -----------------------------
dv = DataValidation(type="list", formula1="=LISTE_DOMAINS", allow_blank=True)
dv.prompt = "Choisis un domaine"
dv.promptTitle = "Domain"
dv.error = "Valeur non autorisée"
dv.errorTitle = "Erreur"

ws_tdb.add_data_validation(dv)
dv.add("A2")

# -----------------------------
# 5) Zone technique du graphique dans TDB!J:K
# Hypothèse :
# - B = X
# - C = Y
# - D = Domains
# -----------------------------
ws_tdb["J1"] = "X"
ws_tdb["K1"] = "Y"

ws_tdb["J2"] = '=FILTER(SCATTER!B2:C1048576,SCATTER!D2:D1048576=$A$2,"")'

# Optionnel : masquer la colonne Z
ws_tdb.column_dimensions["Z"].hidden = True

wb.save("feat-2.xlsx")

Conclusion : ça ne fonctionne pas mais j'ai compris la philosophie : il faut utiliser la fonction FILTER !
FILTER me permet de créer un tableau filtrer à partir de données qui restent non filtrées. Le filtre peut être dans n'importe quelle cellule. Du coup les graphiques doivent se baser sur la plage de filter.
La cellule qui permet de faire le filtre ça va être une cellule de DataValidation

Je vais quand même filtrer depuis SCATTER, ça me semble plus facile que d'aller tout chercher dans data, surtout pour les graphiques

In [ ]:
from openpyxl import load_workbook
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.workbook.defined_name import DefinedName
from openpyxl.utils import get_column_letter

file_path = "../output/test/test.xlsx"

wb = load_workbook(file_path)
ws_scatter = wb["SCATTER"]
ws_tdb = wb["TDB"]

domains = []
seen = set()

for row in range(2, ws_scatter.max_row + 1):
    value = ws_scatter[f"D{row}"].value
    if value is not None and str(value).strip() != "":
        # j'ai enlevé le strip avec str(value) pour être sûr d'avoir toutes les différentes combinaisons
        value = str(value)
        if value not in seen:
            seen.add(value)
            domains.append(value)

domains.sort()

start_row = 2
helper_col = 26  # Z
helper_col_letter = get_column_letter(helper_col)

ws_tdb[f"{helper_col_letter}1"] = "LISTE_DOMAINS"
for i, domain in enumerate(domains, start=start_row):
    ws_tdb[f"{helper_col_letter}{i}"] = domain

end_row = start_row + len(domains) - 1

#crée un alias vers ma liste de domaines
if "LISTE_DOMAINS" in wb.defined_names:
    del wb.defined_names["LISTE_DOMAINS"]

ref_domains = f"'TDB'!${helper_col_letter}$2:${helper_col_letter}${end_row}"
wb.defined_names["LISTE_DOMAINS"] = DefinedName("LISTE_DOMAINS", attr_text=ref_domains)

dv = DataValidation(type="list", formula1="=LISTE_DOMAINS", allow_blank=True)
dv.prompt = "Choisis un domaine"
dv.promptTitle = "Domain"
dv.error = "Valeur non autorisée"
dv.errorTitle = "Erreur"

ws_tdb.add_data_validation(dv)
dv.add("A2")

from openpyxl.worksheet.formula import ArrayFormula
#ws_tdb["AA1"] = '=FILTER(SCATTER!A:D,SCATTER!D:D=TDB!A2,'')'
# ArrayFormula("E2:E11", "=SUM(C2:C11*D2:D11)")
formula ='=_xlfn.FILTER(SCATTER!A:D,SCATTER!D:D=TDB!A2,"")'
ws_tdb["AA1"] = ArrayFormula("AA:AD", formula)

wb.save("../output/test/feat-2.xlsx")

Ca fonctionne moyen car openpyxl n'arrive pas à mettre en place la focntion filter nativement.
Du coup je passe par _xlfn et par arrayformula. Dans arrayformula je suis censé mettre la bonne dimension mais comme j'en ai aucune idée j'ai mis les colonnes en entier et ça me met bcp de NA.

Sinon je dis que c'est les 500 premiers ?

In [10]:
from openpyxl.chart import (
    ScatterChart,
    BubbleChart,
    Reference,
    #existe bien
    Series
)

def scatter_chart(
        wb: Workbook, 
        worksheet_chart: Worksheet, 
        worksheet_data: Worksheet, 
        where: str, 
        col_x: int, 
        col_y: int, 
        min_row: int, 
        max_row: int, 
        save_as: str) -> None :
    
    # ajouter la docstring
    # Pas convaincu par le résultat, je ne pense pas que ça fasse réellement un nuage de points, ça relie les points entre eux.
    c1 = ScatterChart()
    # c1.title = "Complexité vs Note utilisateur"
    #c1.style = 10
    c1.legend = None
    c1.y_axis.title = 'Note utilisateur'
    c1.x_axis.title = 'Complexité'
    c1.width = 20
    c1.height = 15
    # c1.x_axis.scaling.min = 5
    # c1.y_axis.scaling.min = 0
    # c1.x_axis.scaling.max = 5
    # c1.y_axis.scaling.max = 10

    x_values = Reference(worksheet_data, min_col=col_x, min_row=min_row, max_row=max_row)
    y_values = Reference(worksheet_data, min_col=col_y, min_row=min_row, max_row=max_row)
    series = Series(y_values, x_values, title_from_data=False)

    series.marker.symbol = "circle" 
    series.marker.size = 5
    series.graphicalProperties.line.noFill = True
    
    c1.series.append(series)

    worksheet_chart.add_chart(c1, where)

    wb.save(save_as)

In [ ]:
scatter_chart(wb,wb["TDB"],wb["TDB"],where="A41",col_x=28,col_y=27,min_row=1,max_row=1000,save_as="../output/test/feat-2.xlsx")

Ca fonctionne mais mes graphiques son pas beaux